In [1]:
import os

In [2]:
pwd

'c:\\Users\\User\\Documents\\chest-cancer-classification\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'c:\\Users\\User\\Documents\\chest-cancer-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir : Path
    base_model_path : Path
    updated_base_model_path : Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int



In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier import logger
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam


[2026-05-30 21:38:31,214: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
]


In [8]:
class ConfigurationManager:
    def __init__(self, config_path = CONFIG_FILE_PATH, params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])
    
    def get_prepare_base_model_config(self):
        prepare_base_model_config = self.config.prepare_base_model

        create_directories([prepare_base_model_config.root_dir])


        return PrepareBaseModelConfig(
            prepare_base_model_config.root_dir,
            prepare_base_model_config.base_model_path, 
            prepare_base_model_config.updated_base_model_path,  
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )


In [9]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [10]:
class PrepareBaseModel:
    def __init__(self, prepare_base_model_config):
        self.config = prepare_base_model_config
    
    def get_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.model.save(self.config.base_model_path)
    
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            model.trainable = False
        elif (freeze_till is not None) and freeze_till > 0:
            for layer in model.layers[:freeze_till]:
                layer.trainable = False
        
        # Building the model using the Sequential API
        full_model = Sequential()
        full_model.add(model)
        full_model.add(Flatten())
        full_model.add(Dense(256, activation="relu"))
        full_model.add(BatchNormalization())
        full_model.add(Dropout(0.5))
        full_model.add(Dense(units=classes, activation="softmax"))

        full_model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        self.full_model.save(self.config.updated_base_model_path)

In [11]:
try:
    configuration_manager = ConfigurationManager()
    base_model_config = configuration_manager.get_prepare_base_model_config()
    base_model = PrepareBaseModel(base_model_config)
    base_model.get_model()
    base_model.update_base_model()
except Exception as e:
    raise e

[2026-05-30 21:38:33,610: INFO : common : line 35 : yaml file: config\config.yaml loaded successfully]
[2026-05-30 21:38:33,617: INFO : common : line 35 : yaml file: params.yaml loaded successfully]
[2026-05-30 21:38:33,620: INFO : common : line 56 : created directory: artifacts]
[2026-05-30 21:38:33,624: INFO : common : line 56 : created directory: artifacts/prepare_base_model]
[2026-05-30 21:38:34,655: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\backend.py:1398: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
]
[2026-05-30 21:38:34,985: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\layers\pooling\max_pooling2d.py:161: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.
]
[2026-05-30 21:38:36,201: WARNING : saving

c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 7, 7, 512)         14714688  
                                                                 
 flatten (Flatten)           (None, 25088)             0         
                                                                 
 dense (Dense)               (None, 256)               6422784   
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 514       
                                                        